# 03LIC_1071 PVLO Alarm Analysis from Events Data

Analyzing alarm behavior for `03LIC_1071` PVLO alarms extracted from the events dataset.

**Goal**: Understand alarm patterns — are these mostly chattering alarms (rapid on/off) or sustained alarms?

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Load events data
events_df = pd.read_csv("../DATA/trip_filtered_events_dedup.csv", low_memory=False)
events_df['VT_Start'] = pd.to_datetime(events_df['VT_Start'])
events_df = events_df.sort_values('VT_Start').reset_index(drop=True)

# Deduplicate: for rows with same (VT_Start, Source, ConditionName, Description),
# merge them by taking the first non-null value per column across duplicates
dedup_cols = ['VT_Start', 'Source', 'ConditionName', 'Description']
pre_dedup = len(events_df)
events_df = events_df.groupby(dedup_cols, sort=False).first().reset_index()
events_df = events_df.sort_values('VT_Start').reset_index(drop=True)
print(f"Deduplication: {pre_dedup} → {len(events_df)} rows (removed {pre_dedup - len(events_df)} duplicates)")
events_df.to_csv("../DATA/trip_filtered_events_dedup.csv", index=False)
events_df = events_df[events_df['VT_Start'] >= pd.to_datetime("2022-01-01")]

# Filter: 03LIC_1071 PVLO, Category=1 (actual alarm events, not ACK/SHELVE)
pvlo = events_df[
    (events_df['Source'] == '03LIC_1071') &
    (events_df['ConditionName'] == 'PVLO') &
    (events_df['Category'] == 1)
].copy()

print(f"Total PVLO Category=1 events: {len(pvlo)}")
print(f"  Alarm starts (Action is NaN/blank): {pvlo['Action'].isna().sum().sum()}")
print(f"  Alarm ends (Action = OK): {(pvlo['Action'] == 'OK').sum()}")
print(f"  Date range: {pvlo['VT_Start'].min()} to {pvlo['VT_Start'].max()}")
print(f"\nAction breakdown:")
print(pvlo['Action'].value_counts(dropna=False))

Deduplication: 1364694 → 1364694 rows (removed 0 duplicates)
Total PVLO Category=1 events: 2770
  Alarm starts (Action is NaN/blank): 1381
  Alarm ends (Action = OK): 1389
  Date range: 2022-01-05 08:53:41.852900 to 2025-06-22 17:14:56.204800

Action breakdown:
Action
OK      1389
None    1381
Name: count, dtype: int64


In [11]:
events_df[events_df['Description'].str.contains(r'MOD*', na=False)]['Description'].value_counts()

Description
MODE                                                                                                  4905
HYP1COM MOS ACTIVE                                                                                     296
OFFNORM                LOW      HYP1COM MOS ACTIVE       1I                          1          31      93
HYP1COM UNIT MOS                                                                                        71
3K101 MCC MOTOR OVERLOAD                                                                                49
3K151 UNIT MOS                                                                                          45
OFFNORM                LOW      HYP1COM UNIT MOS         1K                          1          31      31
3ER1BD UNIT MOS                                                                                         29
OFFNORM                LOW      3K151 UNIT MOS           1I                          1          31      20
3ER1AD UNIT MOS          

In [17]:
events_df[events_df['Source'] == '03LIC_1071'].head(20)

,VT_Start,Source,ConditionName,Description,Action,Actor,AreaName,AlarmLimit,Block,Category,...,ShelvedReason,SourceParameter,Station,Time,TransactionID,Units,Value,H,TagID,AlarmStatus
124758,2022-01-05 08:42:28.203100,03LIC_1071,CHANGE,SP,None,None,1F,NaN,NaN,7,...,None,NaN,None,132858313482031000,35633626,%,20.0000,2022_01_05_10,03LIC_1071,None
124794,2022-01-05 08:45:10.529100,03LIC_1071,CHANGE,SP,None,None,1F,NaN,NaN,7,...,None,NaN,None,132858315105291000,35634347,%,10.0000,2022_01_05_10,03LIC_1071,None
124852,2022-01-05 08:52:21.353100,03LIC_1071,CHANGE,SP,None,None,1F,NaN,NaN,7,...,None,NaN,None,132858319413531000,35636223,%,15.0000,2022_01_05_10,03LIC_1071,None
124864,2022-01-05 08:53:41.852900,03LIC_1071,PVLO,3E107 LEVEL,None,None,1F,28.75,NaN,1,...,None,NaN,None,132858320218529000,35636569,None,28.724,2022_01_05_10,03LIC_1071,ENABLE
124965,2022-01-05 09:03:26.605000,03LIC_1071,CHANGE,SP,None,None,1F,NaN,NaN,7,...,None,NaN,None,132858326066050000,35639228,%,20.0000,2022_01_05_10,03LIC_1071,None
124972,2022-01-05 09:04:27.410400,03LIC_1071,CHANGE,SP,None,None,1F,NaN,NaN,7,...,None,NaN,None,132858326674104000,35639488,%,22.0000,2022_01_05_10,03LIC_1071,None
124973,2022-01-05 09:04:29.505700,03LIC_1071,CHANGE,SP,None,None,1F,NaN,NaN,7,...,None,NaN,None,132858326695057000,35639499,%,24.0000,2022_01_05_10,03LIC_1071,None
125035,2022-01-05 09:09:38.567300,03LIC_1071,PVLO,PVLO LOW 3E107 LEVEL ...,ACK,None,1F,NaN,NaN,7,...,None,NaN,None,132858329785673000,35640748,None,None,2022_01_05_10,03LIC_1071,None
125076,2022-01-05 09:13:24.853000,03LIC_1071,CHANGE,SP,None,None,1F,NaN,NaN,7,...,None,NaN,None,132858332048530000,35641265,%,25.0000,2022_01_05_10,03LIC_1071,None
125093,2022-01-05 09:15:55.323100,03LIC_1071,CHANGE,SP,None,None,1F,NaN,NaN,7,...,None,NaN,None,132858333553231000,35641605,%,27.0000,2022_01_05_10,03LIC_1071,None


In [18]:
events_df.columns

Index(['VT_Start', 'Source', 'ConditionName', 'Description', 'Action', 'Actor',
       'AreaName', 'AlarmLimit', 'Block', 'Category', 'EventID', 'Flags',
       'LocalTime', 'LocationFullName', 'LocationTagName', 'PrevValue',
       'Priority', 'ReceivedDelay', 'ServerName', 'ShelvedReason',
       'SourceParameter', 'Station', 'Time', 'TransactionID', 'Units', 'Value',
       'H', 'TagID', 'AlarmStatus'],
      dtype='object')

In [19]:
# Extract alarm episodes by walking through events in order
# Rule: Start = Action is NaN, End = Action is OK
# If we see multiple starts in a row, alarm is still ongoing (PV bouncing near threshold)
# Take the FIRST start followed by the NEXT OK as one episode

episodes_list = []
current_start = None
current_start_value = None

for _, row in pvlo.iterrows():
    is_start = pd.isna(row['Action']) or row['Action'] == ''
    is_end = row['Action'] == 'OK'
    
    if is_start and current_start is None:
        current_start = row['VT_Start']
        current_start_value = row['Value']
    elif is_start and current_start is not None:
        pass  # still in alarm
    elif is_end and current_start is not None:
        episodes_list.append({
            'alarm_start': current_start,
            'alarm_end': row['VT_Start'],
            'start_value': current_start_value,
            'end_value': row['Value'],
        })
        current_start = None
        current_start_value = None

episodes = pd.DataFrame(episodes_list)
episodes['episode_num'] = range(1, len(episodes) + 1)
episodes['duration_minutes'] = (episodes['alarm_end'] - episodes['alarm_start']).dt.total_seconds() / 60
episodes['gap_to_next_minutes'] = (
    episodes['alarm_start'].shift(-1) - episodes['alarm_end']
).dt.total_seconds() / 60

print(f"Total alarm episodes: {len(episodes)}")
print(f"Orphan starts (no matching end): {pvlo['Action'].isna().sum() - len(episodes)}")
print(f"\nAlarm DURATION (minutes):")
print(episodes['duration_minutes'].describe().to_string())
print(f"\nGAP to next alarm (minutes):")
print(episodes['gap_to_next_minutes'].dropna().describe().to_string())

Total alarm episodes: 1379
Orphan starts (no matching end): 2

Alarm DURATION (minutes):
count    1379.000000
mean        6.960193
std        20.729632
min         0.008822
25%         2.246327
50%         4.591668
75%         6.802068
max       495.917508

GAP to next alarm (minutes):
count      1378.000000
mean       1314.269327
std        6046.677513
min           0.026467
25%           7.906715
50%          13.500531
75%         201.651889
max      131424.021448


In [20]:
# Classify alarms by duration
def classify_alarm(duration_min):
    if duration_min <= 1:
        return '≤1 min (fleeting)'
    elif duration_min <= 5:
        return '1-5 min (chattering)'
    elif duration_min <= 30:
        return '5-30 min (short)'
    elif duration_min <= 60:
        return '30-60 min (medium)'
    else:
        return '>60 min (sustained)'

episodes['alarm_class'] = episodes['duration_minutes'].apply(classify_alarm)

class_order = ['≤1 min (fleeting)', '1-5 min (chattering)', '5-30 min (short)', 
               '30-60 min (medium)', '>60 min (sustained)']
class_counts = episodes['alarm_class'].value_counts().reindex(class_order).fillna(0).astype(int)
class_pcts = (class_counts / len(episodes) * 100).round(1)

print("Alarm Duration Classification:")
print("=" * 50)
for cls in class_order:
    print(f"  {cls:25s}: {class_counts[cls]:5d} ({class_pcts[cls]:5.1f}%)")
print(f"  {'TOTAL':25s}: {len(episodes):5d}")

Alarm Duration Classification:
  ≤1 min (fleeting)        :   217 ( 15.7%)
  1-5 min (chattering)     :   546 ( 39.6%)
  5-30 min (short)         :   581 ( 42.1%)
  30-60 min (medium)       :    22 (  1.6%)
  >60 min (sustained)      :    13 (  0.9%)
  TOTAL                    :  1379


In [21]:
# Distribution of alarm DURATIONS
fig = make_subplots(rows=2, cols=1, subplot_titles=[
    'Alarm Duration Distribution (all episodes)',
    'Alarm Duration Distribution (zoomed ≤60 min)'
], vertical_spacing=0.12)

fig.add_trace(go.Histogram(x=episodes['duration_minutes'], nbinsx=100, 
                            marker_color='indianred', name='All'), row=1, col=1)
fig.add_trace(go.Histogram(x=episodes[episodes['duration_minutes'] <= 60]['duration_minutes'], 
                            nbinsx=60, marker_color='steelblue', name='≤60 min'), row=2, col=1)

fig.update_xaxes(title_text='Duration (minutes)', row=1, col=1)
fig.update_xaxes(title_text='Duration (minutes)', row=2, col=1)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_yaxes(title_text='Count', row=2, col=1)
fig.update_layout(height=600, showlegend=False, title_text='How long do alarms last?')
fig.show()

In [22]:
# Distribution of GAP between consecutive alarms
gaps = episodes['gap_to_next_minutes'].dropna()

# Classify gaps
def classify_gap(gap_min):
    if gap_min <= 5:
        return '≤5 min (rapid re-alarm)'
    elif gap_min <= 30:
        return '5-30 min'
    elif gap_min <= 60:
        return '30-60 min'
    elif gap_min <= 360:
        return '1-6 hours'
    elif gap_min <= 1440:
        return '6-24 hours'
    else:
        return '>24 hours'

gap_classes = gaps.apply(classify_gap)
gap_order = ['≤5 min (rapid re-alarm)', '5-30 min', '30-60 min', '1-6 hours', '6-24 hours', '>24 hours']
gap_counts = gap_classes.value_counts().reindex(gap_order).fillna(0).astype(int)
gap_pcts = (gap_counts / len(gaps) * 100).round(1)

print("Gap Between Consecutive Alarms:")
print("=" * 50)
for cls in gap_order:
    print(f"  {cls:25s}: {gap_counts[cls]:5d} ({gap_pcts[cls]:5.1f}%)")

fig = make_subplots(rows=2, cols=1, subplot_titles=[
    'Gap to Next Alarm (all)',
    'Gap to Next Alarm (zoomed ≤120 min)'
], vertical_spacing=0.12)

fig.add_trace(go.Histogram(x=gaps, nbinsx=100, marker_color='darkorange', name='All'), row=1, col=1)
fig.add_trace(go.Histogram(x=gaps[gaps <= 120], nbinsx=60, marker_color='teal', name='≤120 min'), row=2, col=1)

fig.update_xaxes(title_text='Gap to next alarm (minutes)', row=1, col=1)
fig.update_xaxes(title_text='Gap to next alarm (minutes)', row=2, col=1)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_yaxes(title_text='Count', row=2, col=1)
fig.update_layout(height=600, showlegend=False, title_text='How quickly does the alarm come back?')
fig.show()

Gap Between Consecutive Alarms:
  ≤5 min (rapid re-alarm)  :   294 ( 21.3%)
  5-30 min                 :   546 ( 39.6%)
  30-60 min                :    87 (  6.3%)
  1-6 hours                :   141 ( 10.2%)
  6-24 hours               :   126 (  9.1%)
  >24 hours                :   184 ( 13.4%)


In [23]:
# Filter to 2025 and build clusters (30 min gap threshold)
CLUSTER_GAP_THRESHOLD = 30  # minutes

episodes_2025 = episodes.copy().reset_index(drop=True)
# episodes_2025 = episodes[episodes['alarm_start'].dt.year == 2025].copy().reset_index(drop=True)
print(f"2025 alarm episodes: {len(episodes_2025)} (out of {len(episodes)} total)")

# Recompute gaps for 2025
episodes_2025['gap_to_next_minutes'] = (
    episodes_2025['alarm_start'].shift(-1) - episodes_2025['alarm_end']
).dt.total_seconds() / 60

# Assign cluster_id: consecutive alarms with gap ≤ threshold belong to the same cluster
cluster_id = 0
cluster_ids = [0]
for g in episodes_2025['gap_to_next_minutes'].iloc[:-1]:
    if pd.notna(g) and g <= CLUSTER_GAP_THRESHOLD:
        cluster_ids.append(cluster_id)
    else:
        cluster_id += 1
        cluster_ids.append(cluster_id)

episodes_2025['cluster_id'] = cluster_ids

# Compute cluster-level stats
clusters = episodes_2025.groupby('cluster_id').agg(
    cluster_start=('alarm_start', 'min'),
    cluster_end=('alarm_end', 'max'),
    n_alarms=('episode_num', 'count')
).sort_values('cluster_start')

clusters['total_duration_min'] = (clusters['cluster_end'] - clusters['cluster_start']).dt.total_seconds() / 60
clusters['gap_to_next_cluster_min'] = (
    clusters['cluster_start'].shift(-1) - clusters['cluster_end']
).dt.total_seconds() / 60

# Classify cluster types
def classify_cluster(row):
    if row['n_alarms'] == 1 and row['total_duration_min'] <= 5:
        return 'Isolated brief alarm'
    elif row['n_alarms'] <= 3 and row['total_duration_min'] <= 30:
        return 'Small cluster'
    elif row['total_duration_min'] <= 120:
        return 'Medium situation (<2h)'
    else:
        return 'Extended situation (>2h)'

clusters['cluster_type'] = clusters.apply(classify_cluster, axis=1)

# Summary
print(f"\n{len(episodes_2025)} raw alarms → {len(clusters)} independent clusters (gap threshold: {CLUSTER_GAP_THRESHOLD} min)")
print(f"  Single-alarm: {(clusters['n_alarms'] == 1).sum()}")
print(f"  Multi-alarm: {(clusters['n_alarms'] > 1).sum()}")
print(f"\nCluster sizes (# alarms per cluster):")
print(clusters['n_alarms'].describe().to_string())
print(f"\nGap between clusters (minutes):")
print(clusters['gap_to_next_cluster_min'].dropna().describe().to_string())
print(f"\nCluster types:")
for ctype in ['Isolated brief alarm', 'Small cluster', 'Medium situation (<2h)', 'Extended situation (>2h)']:
    count = (clusters['cluster_type'] == ctype).sum()
    pct = count / len(clusters) * 100
    avg_alarms = clusters[clusters['cluster_type'] == ctype]['n_alarms'].mean()
    avg_dur = clusters[clusters['cluster_type'] == ctype]['total_duration_min'].mean()
    print(f"  {ctype:30s}: {count:4d} ({pct:5.1f}%) | avg {avg_alarms:.1f} alarms, avg {avg_dur:.0f} min")

2025 alarm episodes: 1379 (out of 1379 total)

1379 raw alarms → 539 independent clusters (gap threshold: 30 min)
  Single-alarm: 334
  Multi-alarm: 205

Cluster sizes (# alarms per cluster):
count    539.000000
mean       2.558442
std        5.437873
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max       87.000000

Gap between clusters (minutes):
count       538.000000
mean       3352.594020
std        9323.421719
min          30.429168
25%          95.983968
50%         735.255382
75%        2676.261320
max      131424.021448

Cluster types:
  Isolated brief alarm          :  187 ( 34.7%) | avg 1.0 alarms, avg 3 min
  Small cluster                 :  202 ( 37.5%) | avg 1.4 alarms, avg 13 min
  Medium situation (<2h)        :  116 ( 21.5%) | avg 4.1 alarms, avg 59 min
  Extended situation (>2h)      :   34 (  6.3%) | avg 13.1 alarms, avg 206 min


In [24]:
# Distribution of gaps between clusters
cluster_gaps = clusters['gap_to_next_cluster_min'].dropna()

def classify_cluster_gap(gap_min):
    if gap_min <= 60:
        return '≤1 hour'
    elif gap_min <= 360:
        return '1-6 hours'
    elif gap_min <= 1440:
        return '6-24 hours'
    elif gap_min <= 4320:
        return '1-3 days'
    elif gap_min <= 10080:
        return '3-7 days'
    else:
        return '>7 days'

gap_cls = cluster_gaps.apply(classify_cluster_gap)
gap_order = ['≤1 hour', '1-6 hours', '6-24 hours', '1-3 days', '3-7 days', '>7 days']
gap_counts = gap_cls.value_counts().reindex(gap_order).fillna(0).astype(int)
gap_pcts = (gap_counts / len(cluster_gaps) * 100).round(1)

print(f"Gap Between Alarm Clusters ({CLUSTER_GAP_THRESHOLD}-min threshold, 2025):")
print("=" * 60)
for cls in gap_order:
    print(f"  {cls:20s}: {gap_counts[cls]:5d} ({gap_pcts[cls]:5.1f}%)")

fig = make_subplots(rows=2, cols=1, subplot_titles=[
    'Gap Between Clusters — All (2025)',
    'Gap Between Clusters — Zoomed ≤1440 min / 24h (2025)'
], vertical_spacing=0.15)

fig.add_trace(go.Histogram(x=cluster_gaps, nbinsx=80, marker_color='mediumpurple'), row=1, col=1)
fig.add_trace(go.Histogram(x=cluster_gaps[cluster_gaps <= 1440], nbinsx=60, marker_color='mediumseagreen'), row=2, col=1)

fig.update_xaxes(title_text='Gap (minutes)', row=1, col=1)
fig.update_xaxes(title_text='Gap (minutes)', row=2, col=1)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_yaxes(title_text='Count', row=2, col=1)
fig.update_layout(height=600, showlegend=False, 
                  title_text=f'Gap between alarm clusters ({CLUSTER_GAP_THRESHOLD}-min threshold, 2025)')
fig.show()

Gap Between Alarm Clusters (30-min threshold, 2025):
  ≤1 hour             :    87 ( 16.2%)
  1-6 hours           :   141 ( 26.2%)
  6-24 hours          :   126 ( 23.4%)
  1-3 days            :    98 ( 18.2%)
  3-7 days            :    48 (  8.9%)
  >7 days             :    38 (  7.1%)


In [25]:
# Try multiple gap thresholds to find truly independent alarm situations
for threshold in [5, 15, 30, 60, 120]:
    cid = 0
    cids = [0]
    for g in episodes_2025['gap_to_next_minutes'].iloc[:-1]:
        if pd.notna(g) and g <= threshold:
            cids.append(cid)
        else:
            cid += 1
            cids.append(cid)
    n_clusters = cid + 1
    # Compute inter-cluster gaps for this threshold
    episodes_2025[f'_cid_{threshold}'] = cids
    cd = episodes_2025.groupby(f'_cid_{threshold}').agg(
        start=('alarm_start', 'min'), end=('alarm_end', 'max'),
        n_alarms=('episode_num', 'count')
    ).sort_values('start')
    cd['gap_next'] = (cd['start'].shift(-1) - cd['end']).dt.total_seconds() / 60
    median_gap = cd['gap_next'].dropna().median()
    pct_under_30 = (cd['gap_next'].dropna() <= 30).mean() * 100
    
    print(f"Threshold {threshold:>3d} min → {n_clusters:>4d} clusters | "
          f"median inter-cluster gap: {median_gap:>8.1f} min | "
          f"% clusters with gap ≤30 min: {pct_under_30:.1f}%")

Threshold   5 min → 1085 clusters | median inter-cluster gap:     28.6 min | % clusters with gap ≤30 min: 50.4%


Threshold  15 min →  632 clusters | median inter-cluster gap:    324.2 min | % clusters with gap ≤30 min: 14.7%
Threshold  30 min →  539 clusters | median inter-cluster gap:    735.3 min | % clusters with gap ≤30 min: 0.0%
Threshold  60 min →  452 clusters | median inter-cluster gap:   1156.4 min | % clusters with gap ≤30 min: 0.0%
Threshold 120 min →  388 clusters | median inter-cluster gap:   1342.1 min | % clusters with gap ≤30 min: 0.0%


In [26]:
# Build output dataframe: one row per raw alarm, with cluster info
output = episodes_2025[['episode_num', 'alarm_start', 'alarm_end', 'duration_minutes', 
                         'gap_to_next_minutes', 'start_value', 'end_value', 'cluster_id']].copy()

# Add cluster-level info
output = output.merge(
    clusters[['n_alarms', 'cluster_start', 'cluster_end', 'total_duration_min', 'gap_to_next_cluster_min', 'cluster_type']],
    left_on='cluster_id', right_index=True
)
output = output.rename(columns={
    'n_alarms': 'cluster_total_alarms',
    'cluster_start': 'cluster_start_time',
    'cluster_end': 'cluster_end_time',
    'total_duration_min': 'cluster_total_duration_min'
})

# Renumber clusters from 1
cluster_id_map = {old: new for new, old in enumerate(sorted(output['cluster_id'].unique()), 1)}
output['cluster_id'] = output['cluster_id'].map(cluster_id_map)
output = output.sort_values('alarm_start').reset_index(drop=True)

# --- Extract operator control actions (CHANGE events) per cluster ---
# Window: [cluster_start - 30 min, cluster_end + 30 min]
WINDOW_MARGIN = pd.Timedelta(minutes=30)

change_events = events_df[
    events_df['ConditionName'] == 'CHANGE'
].copy()

# Build cluster lookup with renumbered IDs
cluster_lookup = clusters.copy()
cluster_lookup['cluster_id_new'] = cluster_lookup.index.map(cluster_id_map)
cluster_lookup = cluster_lookup.sort_values('cluster_start').reset_index(drop=True)

actions_list = []
for _, cl in cluster_lookup.iterrows():
    window_start = cl['cluster_start'] - WINDOW_MARGIN
    window_end = cl['cluster_end'] + WINDOW_MARGIN
    
    mask = (change_events['VT_Start'] >= window_start) & (change_events['VT_Start'] <= window_end)
    cluster_actions = change_events[mask].copy()
    cluster_actions['cluster_id'] = cl['cluster_id_new']
    cluster_actions['cluster_start'] = cl['cluster_start']
    cluster_actions['cluster_end'] = cl['cluster_end']
    actions_list.append(cluster_actions)

control_actions = pd.concat(actions_list, ignore_index=True)
control_actions = control_actions.sort_values(['cluster_id', 'VT_Start']).reset_index(drop=True)

# Classify action timing relative to cluster duration
control_actions['action_timing'] = np.where(
    control_actions['VT_Start'] < control_actions['cluster_start'], 'before',
    np.where(control_actions['VT_Start'] > control_actions['cluster_end'], 'after', 'during')
)

# Classify action direction based on Value vs PrevValue
# Use numeric copies to avoid destroying string values like 'MAN'/'AUTO' for MODE changes
_val_num = pd.to_numeric(control_actions['Value'], errors='coerce')
_prev_num = pd.to_numeric(control_actions['PrevValue'], errors='coerce')
control_actions['action_direction'] = np.where(
    _val_num > _prev_num, 'increase',
    np.where(_val_num < _prev_num, 'decrease', 'no_change')
)

print(f"Control actions extracted: {len(control_actions)} CHANGE events across {control_actions['cluster_id'].nunique()} clusters")
print(f"Clusters with no actions: {output['cluster_id'].nunique() - control_actions['cluster_id'].nunique()}")
print(f"\nAction timing breakdown:")
print(control_actions['action_timing'].value_counts().to_string())
print(f"\nTop 10 most acted-on tags:")
print(control_actions['Source'].value_counts().head(10).to_string())

# Save both sheets to Excel
output_path = '../DATA/1071_pvlo_alarms_clustered_with_control_actions.xlsx'
# output_path = '../DATA/1071_pvlo_alarms_2025_clustered_with_control_actions.xlsx'
actions_save_cols = ['cluster_id', 'cluster_start', 'cluster_end', 'action_timing', 'action_direction', 'Source', 'Description', 'VT_Start', 'PrevValue', 'Value']
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    output.to_excel(writer, sheet_name='alarm_clusters', index=False)
    control_actions[actions_save_cols].to_excel(writer, sheet_name='control_actions', index=False)

print(f"\nSaved to {output_path}")
print(f"  Sheet 'alarm_clusters': {len(output)} alarms across {output['cluster_id'].nunique()} clusters")
print(f"  Sheet 'control_actions': {len(control_actions)} actions")
output.head(10)

Control actions extracted: 16094 CHANGE events across 430 clusters
Clusters with no actions: 109

Action timing breakdown:
action_timing
during    7150
before    5983
after     2961

Top 10 most acted-on tags:
Source
03FIC_3435    2308
03PIC_1013    1891
03HIC_1151    1751
03HIC_3100    1669
03LIC_1071    1147
03HIC_1141    1057
03HIC_3132    1045
03PIC_3131     702
03LIC_1034     626
03LIC_3153     551

Saved to ../DATA/1071_pvlo_alarms_clustered_with_control_actions.xlsx
  Sheet 'alarm_clusters': 1379 alarms across 539 clusters
  Sheet 'control_actions': 16094 actions


,episode_num,alarm_start,alarm_end,duration_minutes,gap_to_next_minutes,start_value,end_value,cluster_id,cluster_total_alarms,cluster_start_time,cluster_end_time,cluster_total_duration_min,gap_to_next_cluster_min,cluster_type
0,1,2022-01-05 08:53:41.852900,2022-01-05 09:33:33.104600,39.854195,2901.719147,28.724,31.758,1,1,2022-01-05 08:53:41.852900,2022-01-05 09:33:33.104600,39.854195,2901.719147,Medium situation (<2h)
1,2,2022-01-07 09:55:16.253400,2022-01-07 10:00:27.504300,5.187515,212.969965,28.744,31.754,2,1,2022-01-07 09:55:16.253400,2022-01-07 10:00:27.504300,5.187515,212.969965,Small cluster
2,3,2022-01-07 13:33:25.702200,2022-01-07 13:36:35.204100,3.158365,40.605848,28.742,31.765,3,1,2022-01-07 13:33:25.702200,2022-01-07 13:36:35.204100,3.158365,40.605848,Isolated brief alarm
3,4,2022-01-07 14:17:11.555000,2022-01-07 14:19:30.552800,2.316630,34.883353,28.749,31.765,4,1,2022-01-07 14:17:11.555000,2022-01-07 14:19:30.552800,2.316630,34.883353,Isolated brief alarm
4,5,2022-01-07 14:54:23.554000,2022-01-07 14:58:16.555500,3.883358,297.775787,28.733,31.781,5,1,2022-01-07 14:54:23.554000,2022-01-07 14:58:16.555500,3.883358,297.775787,Isolated brief alarm
5,6,2022-01-07 19:56:03.102700,2022-01-07 20:02:44.103300,6.683343,205.285010,28.746,31.756,6,1,2022-01-07 19:56:03.102700,2022-01-07 20:02:44.103300,6.683343,205.285010,Small cluster
6,7,2022-01-07 23:28:01.203900,2022-01-07 23:31:54.202400,3.883308,3517.398355,28.727,31.791,7,1,2022-01-07 23:28:01.203900,2022-01-07 23:31:54.202400,3.883308,3517.398355,Isolated brief alarm
7,8,2022-01-10 10:09:18.103700,2022-01-10 10:15:30.103100,6.199990,2905.895815,28.735,31.757,8,1,2022-01-10 10:09:18.103700,2022-01-10 10:15:30.103100,6.199990,2905.895815,Small cluster
8,9,2022-01-12 10:41:23.852000,2022-01-12 10:45:23.351600,3.991660,2923.545038,28.743,31.770,9,1,2022-01-12 10:41:23.852000,2022-01-12 10:45:23.351600,3.991660,2923.545038,Isolated brief alarm
9,10,2022-01-14 11:28:56.053900,2022-01-14 11:35:35.057500,6.650060,2442.107422,28.745,31.757,10,1,2022-01-14 11:28:56.053900,2022-01-14 11:35:35.057500,6.650060,2442.107422,Small cluster


In [27]:
output['gap_to_next_cluster_min'].describe()

count      1378.000000
mean       3086.578978
std       10579.062780
min          30.429168
25%         106.718575
50%         627.696683
75%        2651.154967
max      131424.021448
Name: gap_to_next_cluster_min, dtype: float64

In [28]:
# Control action stats for 03LIC_1071, 03LIC_1016, 03PIC_1013 (SP/OP only)
target_tags = ['03LIC_1071', '03LIC_1016', '03PIC_1013']
control_actions['change_magnitude'] = (pd.to_numeric(control_actions['Value'], errors='coerce') - pd.to_numeric(control_actions['PrevValue'], errors='coerce')).abs()

# Filter to SP/OP description events only
sp_op_actions = control_actions[control_actions['Description'].isin(['SP', 'OP'])].copy()
print(f"Total control actions: {len(control_actions)} → SP/OP only: {len(sp_op_actions)}")

for tag in target_tags:
    tag_actions = sp_op_actions[sp_op_actions['Source'] == tag]
    n_prevval_missing = tag_actions['PrevValue'].isna().sum()
    print(f"\n{'='*60}")
    print(f"Tag: {tag} — {len(tag_actions)} SP/OP actions (PrevValue missing: {n_prevval_missing})")
    print(f"{'='*60}")
    
    for desc in ['OP', 'SP']:
        desc_actions = tag_actions[tag_actions['Description'] == desc]
        if len(desc_actions) == 0:
            continue
        print(f"\n  [{desc}] — {len(desc_actions)} actions")
        for d in ['increase', 'decrease', 'no_change']:
            subset = desc_actions[desc_actions['action_direction'] == d]
            n = len(subset)
            if n == 0:
                print(f"    {d}: 0 actions")
                continue
            mag = subset['change_magnitude']
            print(f"    {d}: {n} actions | "
                  f"magnitude — mean: {mag.mean():.2f}, median: {mag.median():.2f}, "
                  f"min: {mag.min():.2f}, max: {mag.max():.2f}, std: {mag.std():.2f}")

Total control actions: 16094 → SP/OP only: 15128

Tag: 03LIC_1071 — 1075 SP/OP actions (PrevValue missing: 0)

  [OP] — 794 actions
    increase: 408 actions | magnitude — mean: 2.78, median: 2.00, min: 0.10, max: 50.00, std: 4.57
    decrease: 357 actions | magnitude — mean: 3.76, median: 2.00, min: 0.10, max: 50.00, std: 7.05
    no_change: 29 actions | magnitude — mean: nan, median: nan, min: nan, max: nan, std: nan

  [SP] — 281 actions
    increase: 189 actions | magnitude — mean: 2.09, median: 2.00, min: 0.01, max: 35.00, std: 3.82
    decrease: 92 actions | magnitude — mean: 3.81, median: 2.00, min: 0.87, max: 34.27, std: 5.44
    no_change: 0 actions

Tag: 03LIC_1016 — 498 SP/OP actions (PrevValue missing: 0)

  [OP] — 335 actions
    increase: 147 actions | magnitude — mean: 4.01, median: 2.00, min: 0.50, max: 30.00, std: 4.32
    decrease: 90 actions | magnitude — mean: 8.04, median: 2.00, min: 0.56, max: 72.78, std: 11.44
    no_change: 98 actions | magnitude — mean: 0.00, m

In [29]:
# Check if any SP/OP rows (all tags) still have PrevValue missing after dedup fix
sp_op_all = control_actions[control_actions['Description'].isin(['SP', 'OP'])]
missing_pv = sp_op_all[sp_op_all['PrevValue'].isna()]

print(f"SP/OP rows with PrevValue missing: {len(missing_pv)} / {len(sp_op_all)} total")
print(f"\nBreakdown by Source:")
print(missing_pv['Source'].value_counts().head(20).to_string())

if len(missing_pv) > 0:
    # Cross-check against raw data
    raw_events = pd.read_csv("../DATA/trip_filtered_events.csv", low_memory=False)
    raw_events['VT_Start'] = pd.to_datetime(raw_events['VT_Start'])
    raw_events['PrevValue'] = pd.to_numeric(raw_events['PrevValue'], errors='coerce')
    raw_change = raw_events[raw_events['ConditionName'] == 'CHANGE']

    found_in_raw = 0
    sample_printed = 0
    for _, row in missing_pv.iterrows():
        raw_matches = raw_change[
            (raw_change['Source'] == row['Source']) &
            (raw_change['VT_Start'] == row['VT_Start']) &
            (raw_change['Description'] == row['Description'])
        ]
        if raw_matches['PrevValue'].notna().any():
            found_in_raw += 1
            if sample_printed < 10:
                print(f"  STILL FOUND: {row['Source']} | {row['Description']} | {row['VT_Start']}")
                sample_printed += 1

    print(f"\nSummary: {found_in_raw} / {len(missing_pv)} missing-PrevValue rows have PrevValue in raw data")

SP/OP rows with PrevValue missing: 0 / 15128 total

Breakdown by Source:
Series([], )


In [30]:
control_actions[control_actions['Description'] == 'MODE']['Value'].value_counts()

Value
MAN       401
NORMAL    263
AUTO      122
CAS        59
Name: count, dtype: int64

In [31]:
control_actions[control_actions['Description'] == 'MODE']['PrevValue'].value_counts()

PrevValue
MAN     430
CAS     239
AUTO    176
Name: count, dtype: int64